In [4]:
from pathlib import Path
import pandas as pd

In [7]:
selected_metrics = [
  "mean__chm",
  "max__chm",
  "sd__chm",
  "cv__chm",

  "mean__crr",
  "mean__fhd",
  "mean__veg_height_cv",
  "cv__veg_height_median",
  "mean__veg_height_kurt",
  "sd__crr",
  "sd__vci",
  "mean__vci",

  # "mean__groundstorey_capture",
  # "mean__understorey_capture",
  # "mean__midstorey_capture",
  # "mean__upperstorey_capture",
  # "sd__groundstorey_capture",
  # "sd__understorey_capture",
  # "sd__midstorey_capture",
  # "sd__upperstorey_capture",

  "mean__canopy_cover_gt1m",
  "sd__canopy_cover_gt1m"
]


In [10]:
csv_dir = Path("../csvs")
plot_summaries = pd.read_csv(csv_dir / "plot_all_metrics.csv")
plot_summaries['site'] = plot_summaries['id'].str[0:-3]
plot_summaries = plot_summaries.set_index('id')
plot_summaries = plot_summaries[['site', *selected_metrics]]

site_info = pd.read_csv(csv_dir / "site_info.csv")
site_info = site_info.set_index('site_id')
site_info = site_info.drop(columns=['Unnamed: 0'])
site_info['year_estab'] = site_info['year_estab'].fillna(1950)
site_info['years_since_dist'] = 2025 - site_info['year_estab']
site_info = site_info[['site_type', 'year_estab', 'years_since_dist', 'elev_mean', 'slope_mean']]
site_info['forest_type'] = site_info['site_type'].str[0:2]


# # Join site_info onto plot_summaries using the site column
plot_representative_metrics = plot_summaries.reset_index().merge(site_info.reset_index(), left_on='site', right_on='site_id', how='left')

plot_representative_metrics.to_csv(csv_dir / "plot_representative_metrics.csv")

plot_representative_metrics.columns


Index(['id', 'site', 'mean__chm', 'max__chm', 'sd__chm', 'cv__chm',
       'mean__crr', 'mean__fhd', 'mean__veg_height_cv',
       'cv__veg_height_median', 'mean__veg_height_kurt', 'sd__crr', 'sd__vci',
       'mean__vci', 'mean__canopy_cover_gt1m', 'sd__canopy_cover_gt1m',
       'site_id', 'site_type', 'year_estab', 'years_since_dist', 'elev_mean',
       'slope_mean', 'forest_type'],
      dtype='object')

In [11]:
# Create site-level representative metrics by grouping by site and taking the mean
# First, identify the metric columns (excluding non-metric columns)
non_metric_columns = ['id', 'site', 'site_id', 'site_type', 'year_estab', 'years_since_dist', 'elev_mean', 'slope_mean', 'forest_type']
metric_columns = [col for col in plot_representative_metrics.columns if col not in non_metric_columns]

# Group by site and calculate mean for metric columns
site_representative_metrics = plot_representative_metrics.groupby('site')[metric_columns].mean().reset_index()

# Add back the site information (taking the first value for each site since they should be the same)
site_info_cols = ['site_type', 'year_estab', 'years_since_dist', 'elev_mean', 'slope_mean', 'forest_type']
site_info_summary = plot_representative_metrics.groupby('site')[site_info_cols].first().reset_index()

# Merge the metric means with site information
site_representative_metrics = site_representative_metrics.merge(site_info_summary, on='site', how='left')

# Reorder columns to have site info first, then metrics
ordered_columns = ['site'] + site_info_cols + metric_columns
site_representative_metrics = site_representative_metrics[ordered_columns]

# Export to CSV
site_representative_metrics.to_csv(csv_dir / "site_representative_metrics.csv", index=False)

site_representative_metrics

,site,site_type,year_estab,years_since_dist,elev_mean,slope_mean,forest_type,mean__chm,max__chm,sd__chm,...,mean__crr,mean__fhd,mean__veg_height_cv,cv__veg_height_median,mean__veg_height_kurt,sd__crr,sd__vci,mean__vci,mean__canopy_cover_gt1m,sd__canopy_cover_gt1m
0,ULO_212,ULO,1950.0,75.0,403.702572,17.387889,UL,15.117109,44.558,8.170229,...,0.457862,1.779465,0.583063,0.677572,2.435384,0.212834,0.155862,0.465643,0.739220,0.232440
1,ULY_O_27,ULY,2013.0,12.0,385.852735,14.037288,UL,27.780180,58.780,17.599013,...,0.529815,1.846181,0.708614,0.917863,2.014161,0.204971,0.191542,0.449431,0.561796,0.330902
